In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
#cargamos los datos
euskadi = pd.read_excel("importaciones_exportaciones_euskadi.xlsx",sheet_name="Exportaciones")
puerto = pd.read_csv("datosPuerto.csv", decimal=",")

In [ ]:
euskadi.head()

In [ ]:
puerto.head()

In [ ]:
#Filtramos para que solo sea paises bajos y pasamos las fechas a datetime(años)
euskadi.copy()
euskadi_filtrado = euskadi[(euskadi["País"]== "Países Bajos")]
puerto['FECHA'] = pd.to_datetime(puerto['FECHA'])
puerto['FECHA'] = puerto['FECHA'].dt.year
puerto_filtrado = puerto[(puerto["PAIS_FINAL"] == "NL") | (puerto["PAIS_ORIGEN"] == "NL")]

In [ ]:
euskadi_filtrado.head()

In [ ]:
#Filtramos para que el periodo de años sea del 2009 al 2019(es hasta 2019 poprque el dataset de euskadi solo llega hasta 2019) y remplazamos los valores faltantes por 0
columnas_años = ["2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019"]
euskadi_filtrado[columnas_años] = (euskadi_filtrado[columnas_años].replace("-", 0))
euskadi_filtrado[columnas_años] = euskadi_filtrado[columnas_años].astype(float)


In [ ]:
#Hacemos la suma de las toneladas por año y lo pasamos a kilos
toneladas = euskadi_filtrado[euskadi_filtrado["Unidades"] == "Toneladas"]
importaciones_por_año = toneladas[columnas_años].sum()
importaciones_por_año = importaciones_por_año*1000
print(importaciones_por_año)

In [ ]:
#FIltramos solo paises bajos y españa
#agrupamos los kilos pòr años
#Filtramos solo de 2009 a 2019
puerto_paisesbajos = puerto[(puerto["PAIS_ORIGEN"] == "NL") & (puerto["PAIS_FINAL"] == "ES")]
imp_puerto = puerto_paisesbajos.groupby("FECHA")["PESO_KILOS"].sum()
imp_puerto= imp_puerto.loc[2009:2019]

In [ ]:
imp_puerto

In [ ]:
#Lo pasamos a una serie para que sea mas sencillo el calculo del porcentaje
euskadi_total = pd.Series({
    2009: 704409000,
    2010: 929704000,
    2011: 736005000,
    2012: 744985000,
    2013: 490499000,
    2014: 636977000,
    2015: 802839000,
    2016: 753179000,
    2017: 746275000,
    2018: 531313000,
    2019: 613563000})

puerto_imp = pd.Series({
    2009: 826407028,
    2010: 1096816180,
    2011: 786733730,
    2012: 925922346,
    2013: 732426144,
    2014: 785162500,
    2015: 790342418,
    2016: 690588712,
    2017: 683978612,
    2018: 696356457,
    2019: 831587812})

In [ ]:
#Calculamos la importancia del puerto de bilbao respecto a Euskadi. (Supera el 100% por el Efecto Hinterland).
importaciones = (puerto_imp/euskadi_total)*100
importaciones

### Insight de Negocio: El "Efecto Hinterland"
Observamos que el volumen físico exportado a través del Puerto de Bilbao supera sistemáticamente el 100% del volumen fiscal exportado por la Comunidad Autónoma de Euskadi. Lejos de ser un error en los datos, esto confirma empíricamente la macro-influencia logística del puerto (Efecto Hinterland): Bilbao no solo da salida a la producción vasca, sino que opera como hub logístico central exportando la producción industrial de comunidades limítrofes (Cantabria, La Rioja, Navarra, Burgos, etc.).